In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np 

c:\Users\fathy\anaconda3\envs\ai-engineering\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset('jacob-hugging-face/job-descriptions', split='train')
df = pd.DataFrame(dataset)
print('dataframe loaded')

dataframe loaded


In [3]:
df

,company_name,job_description,position_title,description_length,model_response
0,Google,minimum qualifications\nbachelors degree or eq...,Sales Specialist,2727,"{\n ""Core Responsibilities"": ""Responsible fo..."
1,Apple,description\nas an asc you will be highly infl...,Apple Solutions Consultant,828,"{\n ""Core Responsibilities"": ""as an asc you ..."
2,Netflix,its an amazing time to be joining netflix as w...,Licensing Coordinator - Consumer Products,3205,"{\n ""Core Responsibilities"": ""Help drive bus..."
3,Robert Half,description\n\nweb designers looking to expand...,Web Designer,2489,"{\n ""Core Responsibilities"": ""Designing webs..."
4,TrackFive,at trackfive weve got big goals were on a miss...,Web Developer,3167,"{\n ""Core Responsibilities"": ""Build and layo..."
...,...,...,...,...,...
848,Menards,job description\n\nparttime\n\nmake big money ...,Management Internship,1122,"{\n ""Core Responsibilities"": ""Responsibiliti..."
849,Parker,responsibilities\nparkers internship program w...,Human Resources Internship - Corporate (Year-...,3840,"{\n ""Core Responsibilities"": ""Assist in gene..."
850,Borgen Project,the borgen project is an innovative national ...,Writer / Journalist Internship,897,"{\n ""Core Responsibilities"": ""Write one arti..."
851,Wyndham Destinations,put the world on vacation\n\nat wyndham destin...,Inbound Customer Service / Sales (Remote),4604,"{\n ""Core Responsibilities"": ""Answer inbound..."


In [4]:
df.dropna(subset=['position_title', 'job_description']).reset_index(drop=True)

,company_name,job_description,position_title,description_length,model_response
0,Google,minimum qualifications\nbachelors degree or eq...,Sales Specialist,2727,"{\n ""Core Responsibilities"": ""Responsible fo..."
1,Apple,description\nas an asc you will be highly infl...,Apple Solutions Consultant,828,"{\n ""Core Responsibilities"": ""as an asc you ..."
2,Netflix,its an amazing time to be joining netflix as w...,Licensing Coordinator - Consumer Products,3205,"{\n ""Core Responsibilities"": ""Help drive bus..."
3,Robert Half,description\n\nweb designers looking to expand...,Web Designer,2489,"{\n ""Core Responsibilities"": ""Designing webs..."
4,TrackFive,at trackfive weve got big goals were on a miss...,Web Developer,3167,"{\n ""Core Responsibilities"": ""Build and layo..."
...,...,...,...,...,...
848,Menards,job description\n\nparttime\n\nmake big money ...,Management Internship,1122,"{\n ""Core Responsibilities"": ""Responsibiliti..."
849,Parker,responsibilities\nparkers internship program w...,Human Resources Internship - Corporate (Year-...,3840,"{\n ""Core Responsibilities"": ""Assist in gene..."
850,Borgen Project,the borgen project is an innovative national ...,Writer / Journalist Internship,897,"{\n ""Core Responsibilities"": ""Write one arti..."
851,Wyndham Destinations,put the world on vacation\n\nat wyndham destin...,Inbound Customer Service / Sales (Remote),4604,"{\n ""Core Responsibilities"": ""Answer inbound..."


In [5]:
df['company_name'] = df['company_name'].fillna("Unknown Company")
df['full_text'] = df['position_title'] + '\n' + df['job_description']

print(f"loaded {len(df)} jobs")

loaded 853 jobs


In [9]:
import chromadb
from chromadb.utils import embedding_functions
client = chromadb.PersistentClient(path="./job_chroma_db")
sentence_transformer = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name= "all-MiniLM-L6-v2",
    device='cuda'
)

collection = client.get_or_create_collection(
    name="agentic_job_search",
    embedding_function= sentence_transformer,
    metadata= {"hnsw:space": "cosine"}
)

batch_size= 5000
total_batches= len(df) // batch_size
print("Embedding and starting documents in chromadb ")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4711.67it/s]


Embedding and starting documents in chromadb 


In [ ]:
if collection.count() == 0:
    for i in range(0, len(df), batch_size):
        # Slice the dataframe for the current batch
        batch_df = df.iloc[i:i+batch_size]
        
        # 1. Documents: The text that will be embedded and searched against
        documents = batch_df['full_text'].tolist()
        
        # 2. Metadata: Additional info for filtering (must be a list of dictionaries)
        metadatas = batch_df[['position_title', 'company_name', 'location']].to_dict('records')
        
        # 3. IDs: Unique identifier for each document in the vector store
        ids = [f"job_{idx}" for idx in batch_df.index]
        
        # Add the batch to ChromaDB
        collection.add(
            documents=documents,
            metadatas=metadatas,
            ids=ids
        )
        
        print(f"Inserted batch {i//batch_size + 1}/{total_batches + 1}")
        
print(f"Chromadb collection ready with {collection.count()} jobs")

Chromadb collection ready with 853 jobs


In [14]:
test_query = "looking for a fat-track MLOps and AL Engineer role"

results = collection.query(
    query_texts=[test_query],
    n_results=3
)

print('top result title', results['metadatas'][0][0]['position_title'])
print('top result company', results['metadatas'][0][0]['company_name'])


top result title R0049: PM Full Time Material Handler
top result company FedEx Express


pip install langchain-ollama

In [35]:
%%writefile app.py
import chromadb
from chromadb.utils import embedding_functions
import streamlit as st
from chromadb.utils import embedding_functions
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

st.set_page_config(page_title="Agentic job Search", page_icon="🚀", layout="wide")

@st.cache_resource
def load_db():
    client = chromadb.PersistentClient(path="./job_chroma_db")
    sentence_transformer_ef =embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name = "all-MiniLM-L6-v2",
    )
    collection = client.get_or_create_collection(
        name = "agentic_job_search",
        embedding_function = sentence_transformer_ef
    )
    return collection

@st.cache_resource
def load_llm():
    llm = ChatOllama(model="llama3.1:8b", tempreature=0.3)
    prompt = ChatPromptTemplate.from_template(
        template= """  
        the user is looking for a job with these characteristics :"{query}"
        you found this job :"{job_title}"
        Description snippet: {job_desc}
        in exactly 2 short, punchy sentences, explain to the user WHY this specific job is a good match for what they are looking for. Focus on the skills and intent    """
    )
    return llm, prompt

collection = load_db()
llm, prompt = load_llm()

st.title("Agentic job search engine (local)")
st.markdown("search via vector database (chromadb) + local llama3 ")

with st.sidebar:
    st.header("search parameter")
    top_k = st.slider("results to fetch ", min_value=1, max_value=10, value=3)
    fillter_location = st.text_input("filter by location ", placeholder="e.g, cairo, new york")

query = st.text_input("describe your ideal rule, skills, and goals:", "building RAG pipeline, deploying local llms, and python engineering")

if query:
    with st.spinner("Searching vector space..."):
        where_clause=None
        if fillter_location:
            where_clause= {"location": {"$contains": fillter_location}}

        results =collection.query(
            query_texts= [query],
            n_results= top_k,
            where= where_clause
        )

    st.success(f"retrieved top{top_k} matches ")


    for i in range(top_k):
        meta = results['metadatas'][0][i]
        doc =results['documents'][0][i]
        job_title = meta['position_title']
        company = meta['company_name']


        formatted_prompt = prompt.format(query=query, job_title=job_title, job_desc=doc[:1000])
        rationale = llm.invoke(formatted_prompt).content


        with st.container():
            st.subheader(f"{job_title} at {company}")
            st.info(f"why its a match {rationale}")

            with st.expander("view full job describtion"):
                st.write(doc)
            st.markdown("---")

Overwriting app.py
